# Homework 04: Data Acquisition and Ingestion

This notebook builds two reproducible raw-data ingestion workflows and validates each result before saving it.

## Sources and parameters

1. **API-style market pull:** Yahoo Finance through the `yfinance` client. Ticker, period, and interval are read from `.env` (defaults: SPY, one year, daily). No API key is required.
2. **HTML table scrape:** [Wikipedia's S&P 500 companies page](https://en.wikipedia.org/wiki/List_of_S%26P_500_companies), table `#constituents`. The request uses a timeout, status check, user-agent, and a fallback selector.

Validation covers required columns, shape, duplicate keys, missing values, parsed dates/numbers, and basic market-price rules. Raw filenames include a UTC run timestamp.

In [1]:
from datetime import datetime, timezone
import os
from pathlib import Path

from bs4 import BeautifulSoup
from dotenv import load_dotenv
import pandas as pd
import requests
import yfinance as yf

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
RAW_DIR.mkdir(parents=True, exist_ok=True)
load_dotenv(PROJECT_ROOT / '.env')
RUN_TS = datetime.now(timezone.utc).strftime('%Y%m%d-%H%M')
print(f'Project root: {PROJECT_ROOT}')
print(f'UTC run timestamp: {RUN_TS}')

Project root: /Users/zhanyadan/Desktop/bootcamp_Yadan_Zhan/homework/homework04
UTC run timestamp: 20260819-1659


## 1. Market data ingestion

In [2]:
ticker = os.getenv('MARKET_TICKER', 'SPY').upper()
period = os.getenv('MARKET_PERIOD', '1y')
interval = os.getenv('MARKET_INTERVAL', '1d')

try:
    market_df = yf.download(
        ticker, period=period, interval=interval, auto_adjust=False,
        progress=False, timeout=30
    )
except Exception as exc:
    raise RuntimeError(f'Yahoo Finance request failed for {ticker}') from exc

if market_df.empty:
    raise ValueError(f'Yahoo Finance returned no rows for {ticker}')
if isinstance(market_df.columns, pd.MultiIndex):
    market_df.columns = market_df.columns.get_level_values(0)
market_df = market_df.reset_index()
market_df.columns.name = None
market_df['Date'] = pd.to_datetime(market_df['Date'], errors='coerce', utc=True)
numeric_cols = ['Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume']
for column in numeric_cols:
    market_df[column] = pd.to_numeric(market_df[column], errors='coerce')
market_df.head()

,Date,Adj Close,Close,High,Low,Open,Volume
0,2025-08-19 00:00:00+00:00,632.798401,639.809998,644.109985,638.479980,643.119995,69750700
1,2025-08-20 00:00:00+00:00,631.117004,638.109985,639.659973,632.950012,639.400024,88890300
2,2025-08-21 00:00:00+00:00,628.585083,635.549988,637.969971,633.809998,636.280029,54805800
3,2025-08-22 00:00:00+00:00,638.238220,645.309998,646.500000,637.250000,637.760010,84083200
4,2025-08-25 00:00:00+00:00,635.429199,642.469971,645.289978,642.349976,644.039978,51274300


In [3]:
market_required = {'Date', 'Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume'}
missing_market_columns = market_required.difference(market_df.columns)
assert not missing_market_columns, f'Missing columns: {missing_market_columns}'
assert market_df.shape[0] > 0, 'Market dataset has no rows'
assert market_df['Date'].notna().all(), 'Some market dates did not parse'
assert not market_df['Date'].duplicated().any(), 'Duplicate market dates found'
assert (market_df[['Open', 'High', 'Low', 'Close', 'Adj Close']] > 0).all().all(), 'Prices must be positive'
assert (market_df['High'] >= market_df['Low']).all(), 'High must be at least Low'

market_na = market_df[list(market_required)].isna().sum().sort_index()
print(f'Market shape: {market_df.shape}')
print('Market NA counts:')
print(market_na.to_string())
market_path = RAW_DIR / f'api_yahoo_{ticker}_{RUN_TS}.csv'
market_df.to_csv(market_path, index=False)
print(f'Saved: {market_path}')

Market shape: (252, 7)
Market NA counts:
Adj Close    0
Close        0
Date         0
High         0
Low          0
Open         0
Volume       0
Saved: /Users/zhanyadan/Desktop/bootcamp_Yadan_Zhan/homework/homework04/data/raw/api_yahoo_SPY_20260819-1659.csv


## 2. Small-table scraping ingestion

In [4]:
source_url = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'
headers = {'User-Agent': 'Mozilla/5.0 (compatible; coursework-data-ingestion/1.0)'}
try:
    response = requests.get(source_url, headers=headers, timeout=30)
    response.raise_for_status()
except requests.RequestException as exc:
    raise RuntimeError('Could not download the S&P 500 constituents page') from exc

soup = BeautifulSoup(response.text, 'html.parser')
table = soup.select_one('table#constituents') or soup.select_one('table.wikitable')
if table is None:
    raise ValueError('Could not find the constituents table')

table_rows = table.select('tr')
if not table_rows:
    raise ValueError('The constituents table contains no rows')
column_names = [cell.get_text(' ', strip=True) for cell in table_rows[0].select('th, td')]
rows = []
for row in table_rows[1:]:
    values = [cell.get_text(' ', strip=True) for cell in row.select('th, td')]
    if len(values) == len(column_names):
        rows.append(values)

constituents_df = pd.DataFrame(rows, columns=column_names)
constituents_df['Date added'] = pd.to_datetime(constituents_df['Date added'], errors='coerce')
constituents_df['CIK'] = pd.to_numeric(constituents_df['CIK'], errors='coerce').astype('Int64')
constituents_df.head()

,Symbol,Security,GICS Sector,GICS Sub-Industry,Headquarters Location,Date added,CIK,Founded
0,MMM,3M,Industrials,Industrial Conglomerates,"Saint Paul, Minnesota",1957-03-04,66740,1902
1,AOS,A. O. Smith,Industrials,Building Products,"Milwaukee , Wisconsin",2017-07-26,91142,1916
2,ABT,Abbott Laboratories,Health Care,Health Care Equipment,"North Chicago, Illinois",1957-03-04,1800,1888
3,ABBV,AbbVie,Health Care,Biotechnology,"North Chicago, Illinois",2012-12-31,1551152,2013 (1888)
4,ACN,Accenture,Information Technology,IT Consulting & Other Services,"Dublin , Ireland",2011-07-06,1467373,1989


In [5]:
scrape_required = {'Symbol', 'Security', 'GICS Sector', 'GICS Sub-Industry', 'CIK'}
missing_scrape_columns = scrape_required.difference(constituents_df.columns)
assert not missing_scrape_columns, f'Missing columns: {missing_scrape_columns}'
assert constituents_df.shape[0] >= 400, 'Unexpectedly small constituents table'
assert constituents_df['Symbol'].str.strip().ne('').all(), 'Blank symbols found'
assert not constituents_df['Symbol'].duplicated().any(), 'Duplicate symbols found'
assert constituents_df['CIK'].notna().all(), 'CIK must be numeric and non-missing'

scrape_na = constituents_df[list(scrape_required)].isna().sum().sort_index()
print(f'Scraped-table shape: {constituents_df.shape}')
print('Scraped-table NA counts:')
print(scrape_na.to_string())
scrape_path = RAW_DIR / f'scrape_wikipedia_sp500_{RUN_TS}.csv'
constituents_df.to_csv(scrape_path, index=False)
print(f'Saved: {scrape_path}')

Scraped-table shape: (503, 8)
Scraped-table NA counts:
CIK                  0
GICS Sector          0
GICS Sub-Industry    0
Security             0
Symbol               0
Saved: /Users/zhanyadan/Desktop/bootcamp_Yadan_Zhan/homework/homework04/data/raw/scrape_wikipedia_sp500_20260819-1659.csv


## Assumptions and risks

- Yahoo Finance and Wikipedia are external sources whose availability, schema, and content can change without notice.
- SPY is used as a market proxy and does not represent every asset or investor experience.
- The latest trading day may be absent on weekends, holidays, or during temporary provider delays.
- Wikipedia is community maintained. The table is useful for this ingestion exercise but should be cross-checked against an official index source for production decisions.
- The HTML parser uses an ID selector plus a class fallback, but major page redesigns could still require code changes.
- Raw timestamped files preserve what was retrieved at run time. Re-running later may produce different values or constituents.
- `.env` is local and ignored by Git; `.env.example` contains only non-secret parameter names and safe defaults.